In [1]:
import sys
sys.path.insert(0, "/home/jjuradod/qaoa/qaoa_training_pipeline/qaoa_training_pipeline")


In [4]:
!python -c "import qaoa_training_pipeline as p; import qaoa_training_pipeline.training as t; print('PKG:', p.__file__); print('TRAINING:', t.__file__); print('keys:', sorted(t.TRAINERS.keys()))"

PKG: None
TRAINING: None
Traceback (most recent call last):
  File "<string>", line 1, in <module>
AttributeError: module 'qaoa_training_pipeline.training' has no attribute 'TRAINERS'


In [3]:
import qaoa_training_pipeline as p
import qaoa_training_pipeline.training as t
print("pkg:", p.__file__)
print("training:", t.__file__)
print("TRAINERS exists:", hasattr(t, "TRAINERS"))


pkg: None
training: None
TRAINERS exists: False


In [3]:
!pip install -e /home/jjuradod/qaoa/qaoa_training_pipeline

Defaulting to user installation because normal site-packages is not writeable
Obtaining file:///home/jjuradod/qaoa/qaoa_training_pipeline
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for qaoa_training_pipeline (pyproject.toml) ... done
  Created wheel for qaoa_training_pipeline: filename=qaoa_training_pipeline-0.1.0-0.editable-py3-none-any.whl size=7634 sha256=2e7bec0d2d00cd10d2e552a9090977a63d2b52df9a1bf9230af13502444f48fa
  Stored in directory: /tmp/pip-ephem-wheel-cache-jbh2nz0i/wheels/ee/c5/8c/683f779541985fbe045490e8cfe7954137dc854f83582c7610
Successfully built qaoa_training_pipeline
  Attempting uninstall: qaoa_training_pipeline
    Found existing installation: qaoa_training_pipeline 0.1.0
    Uninstalling qaoa_training_pipeline-0.1.0:
      Successfully uninstalled qaoa_training_pipeline-0.1.0


In [6]:
!pip install qiskit_aer

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 53.2 MB/s eta 0:00:0000:01


In [2]:
import os
import subprocess
from pathlib import Path
import json
import numpy as np
from sklearn.cluster import DBSCAN

In [3]:
instances_path = "./instances/line_to_full/"
database_path = "./optimized_angles_database_L2F.json"
instance = "100nodes_2swap_layers.json"
method_path = "./methods/PT_PP_AAA.json"
save_dir = "../../results"

base_path = "../../../QAOA-Parameter-Setting"

root = Path("../qaoa/qaoa_training_pipeline/qaoa_training_pipeline")

In [4]:
files = [
    os.path.join(instances_path, f) for f in os.listdir(instances_path)
    if instance in f and os.path.isfile(os.path.join(instances_path, f))
]

In [ ]:
for file in files:
    file_strings = os.path.basename(file).split("_")
    save_file = f"{file_strings[0]}N{file_strings[1].split('n')[0]}L2S{file_strings[2].split('s')[0]}"
    cmd = [
        "python",
        "-m",
        "train",
        "--input",
        os.path.join(base_path, file),
        "--config",
        os.path.join(base_path,method_path),
        "--save",
        "--save_dir",
        save_dir,
        "--save_file",
        save_file,
    ]
    subprocess.run(cmd, cwd=root)

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/jjuradod/qaoa/qaoa_training_pipeline/qaoa_training_pipeline/train.py", line 36, in <module>
    from qaoa_training_pipeline.training import TRAINERS
ImportError: cannot import name 'TRAINERS' from 'qaoa_training_pipeline.training' (unknown location)
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/jjuradod/qaoa/qaoa_training_pipeline/qaoa_training_pipeline/train.py", line 36, in <module>
    from qaoa_training_pipeline.training import TRAINERS
ImportError: cannot import name 'TRAINERS' from 'qaoa_training_pipeline.training' (unknown location)
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/jjuradod/qaoa/qaoa_training_pipeline/q

In [ ]:
def parse_key(k):

    parts = [p.strip() for p in k.split(",")]
    a = int(parts[0])
    b = float(parts[1])
    c = parts[2] if len(parts) >= 3 else None
    return a, b, c

    raise ValueError(f"Bad key format: {k!r}")

In [ ]:
with open(database_path, "r", encoding="utf-8") as f:
    db = json.load(f)
groups = {}
for k, v in db.items():
    a, b, c = parse_key(k)
    groups.setdefault(a, []).append((k, b, c, v))

In [ ]:
new_db = {}
for a, items in groups.items():
    bs = np.array([it[1] for it in items], dtype=float).reshape(-1, 1)
    labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(bs)

    # collect indices per label
    lab2idx = {}
    for i, lab in enumerate(labels):
        if lab == -1 and not keep_noise:
            continue
        lab2idx.setdefault(int(lab), []).append(i)

    for lab, idxs in lab2idx.items():
        if lab == -1:
            # each noise point -> its own "cluster"
            for i in idxs:
                b0 = float(bs[i, 0])
                center_key = (a, b0)
                k, b, c, v = items[i]
                new_db.setdefault(center_key, []).append(
                    {"orig_key": k, "b": float(b), "c": c, "value": v}
                )
            continue

        b_vals = bs[idxs, 0]
        if center == "mean":
            center_b = float(np.mean(b_vals))
        else:  # "median" default
            center_b = float(np.median(b_vals))

        center_key = (a, center_b)
        for i in idxs:
            k, b, c, v = items[i]
            new_db.setdefault(center_key, []).append(
                {"orig_key": k, "b": float(b), "c": c, "value": v}
            )